# 01 - Theory: Chain Rule, Markov, and the DDM Formulation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yaso09/ddm/blob/master/notebooks/01_Theory.ipynb)

**Summary.** This notebook establishes the mathematical background of the
Distance-Decomposed Model (DDM) and *numerically proves* the key structural
claims:

1. The **chain rule** $P(x_1,\dots,x_n)=\prod_{t=1}^{n} P(x_t\mid x_1,\dots,x_{t-1})$
   is exact and requires no Markov assumption - DDM keeps it in full.
2. The **low-rank pairwise interaction** term
   $F(e_i,e_j) = (U e_i)^\top (V e_j)$ from the theory documents is
   mathematically *identical* to the query-key dot product of standard
   attention (with $W_q = U^\top$, $W_k = V^\top$), so DDM needs no
   separate interaction module.
3. The **distance gate** $g(k)$ is combined with the scores in log-space
   *before* the softmax: $\mathrm{softmax}(s + \log g) = \frac{g\, e^s}{\sum_j g_j e^{s_j}}$,
   i.e. it is equivalent to a *renormalized* post-softmax multiplication,
   but **not** to a plain (non-renormalized) post-softmax multiplication -
   the latter was the failed variant in an early experiment.

The full general form implemented by the model is

$$
H_t = \sum_k \mathrm{softmax}_k\big(s_{t,k} - m_h\cdot k\big)\, g(k)\, e_{t-k}
      \;+\; \gamma\, M_{t-1},
\qquad
z_t = W_o H_t + b,
\qquad
P(x_t = v \mid x_{<t}) = \mathrm{softmax}(z_t)_v ,
$$

where $m_h$ is the per-head ALiBi slope, $g(k)$ the learned (or fixed $1/k$)
distance gate and $M_{t-1}$ the segment memory. Everything below is verified
on toy tensors with `torch.allclose`.

In [1]:
import math
import torch

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.5.1+cu121


## 1. The chain rule is exact (no Markov assumption needed)

Take an arbitrary joint distribution over three tokens and check that
$P(x_1,x_2,x_3) = P(x_1)\,P(x_2\mid x_1)\,P(x_3\mid x_1,x_2)$ holds exactly.
This works for *any* joint distribution - no independence assumption involved.

In [2]:
V = 5
joint = torch.rand(V, V, V) ** 2          # arbitrary positive table
joint = joint / joint.sum()               # normalized -> valid joint P(x1,x2,x3)

p1 = joint.sum(dim=(1, 2))                # P(x1)
p2_g1 = joint.sum(dim=2) / p1.unsqueeze(1)  # P(x2 | x1)
p3_g12 = joint / joint.sum(dim=2, keepdim=True)  # P(x3 | x1, x2)

reconstructed = p1[:, None, None] * p2_g1[:, :, None] * p3_g12
assert torch.allclose(reconstructed, joint, atol=1e-6)
print("chain rule holds exactly:", torch.allclose(reconstructed, joint, atol=1e-6))

# And the autoregressive product form for a sequence x = (1, 2, 3):
x = (1, 2, 3)
chain = p1[x[0]] * p2_g1[x[0], x[1]] * p3_g12[x[0], x[1], x[2]]
print("P(1,2,3) via chain rule =", chain.item())

chain rule holds exactly: True
P(1,2,3) via chain rule = 0.013067788444459438


## 2. The low-rank interaction term is attention

The extended theory document replaces the pairwise interaction
$I_{ij} = F(e_{t-i}, e_{t-j})$ with the low-rank factorization
$F(e_i,e_j) = (U e_i)^\top (V e_j)$, $U, V \in \mathbb{R}^{r\times d}$.

Attention computes $s_{ij} = \frac{(e_i W_q)(e_j W_k)^\top}{\sqrt{d_k}}$.
With $W_q = U^\top$ and $W_k = V^\top$:

$$
(e_i U^\top)(e_j V^\top)^\top = (U e_i)^\top (V e_j) = F(e_i, e_j).
$$

So the interaction term **is** the QK dot product; no extra module exists
in the implementation because the Q/K projections *are* the interaction
term. Verified numerically below (pairwise loop vs. matrix form).

In [3]:
T, d, r = 8, 12, 4
E = torch.randn(T, d)                     # token embeddings
U = torch.randn(r, d)
V = torch.randn(r, d)

# pairwise form: F(e_i, e_j) = (U e_i).T (V e_j)
Ue = E @ U.t()                            # [T, r]
Ve = E @ V.t()                            # [T, r]
F_pairwise = torch.einsum("ir,jr->ij", Ue, Ve)

# attention form: Q K^T with W_q = U.T, W_k = V.T
Wq, Wk = U.t(), V.t()
Q = E @ Wq
K = E @ Wk
F_attention = Q @ K.t()

assert torch.allclose(F_pairwise, F_attention, atol=1e-6)
print("F_pairwise == QK^T:", torch.allclose(F_pairwise, F_attention, atol=1e-6))
print("max abs diff:", (F_pairwise - F_attention).abs().max().item())

F_pairwise == QK^T: True
max abs diff: 3.814697265625e-06


## 3. Pre-softmax log-gate vs. post-softmax multiplication

With a gate $g(k)\in(0,1)$:

$$
\mathrm{softmax}_i(s + \log g)_i = \frac{e^{s_i}\,g_i}{\sum_j e^{s_j} g_j}
$$

which equals the **renormalized** post-softmax multiplication
$g_i\,\mathrm{softmax}_i(s) \,/\, \sum_j g_j\,\mathrm{softmax}_j(s)$.
It does **not** equal the plain (non-renormalized) product
$g_i\,\mathrm{softmax}_i(s)$.

Early experiments found that when the gate was applied after the softmax
(without this renormalization), it degenerated into a correction term that
merely restored the probability mass ALiBi had suppressed, instead of
learning an independent distance signal. DDM therefore adds $\log g(k)$ to
the scores **before** the softmax; the regression test
`tests/test_g_gate_presoftmax.py` locks this behavior.

In [4]:
s = torch.randn(8)                                   # arbitrary scores
g = torch.sigmoid(torch.linspace(-2, 2, 8))          # varying gate

pre = torch.softmax(s + torch.log(g), dim=-1)                       # DDM
post = g * torch.softmax(s, dim=-1)                                 # failed variant
post_renorm = post / post.sum()                                     # equivalent form

print("pre == renormalized post:", torch.allclose(pre, post_renorm, atol=1e-6))
print("pre == plain post:       ", torch.allclose(pre, post, atol=1e-6))
print("row sums - pre:", pre.sum().item(), "| plain post:", post.sum().item())

pre == renormalized post: True
pre == plain post:        False
row sums - pre: 0.9999999403953552 | plain post: 0.5293478965759277


## Summary

- The chain rule is preserved exactly; no Markov truncation is required.
- The pairwise interaction term reduces to the Q/K projection - DDM's
  attention *is* that term.
- The distance gate enters log-additively before the softmax; the
  equivalent renormalized post-softmax form is *not* the naive product,
  and the naive product is the known-failed variant.

Next: `02_Implementation.ipynb` walks through each component on toy inputs.